# UPPP 135 Final Exam

Use your knowledge of Python and urban data science to respond to the following questions using both code and prose. You may need to add code or text cells below each question to add space for your answer. You may use notes, slides, and exercise materials from prior classes.

Uncomment and run the first cell to install the necessary dependencies. Be sure to execute any existing code cells in the notebook as they provide required data and setup. You may collaborate via Discord if you wish.

**Be sure to copy the notebook into your own Drive to avoid possibly losing any work**

In [ ]:
# !pip install geosnap osmnx h3

In [ ]:
import geopandas as gpd
import pandas as pd
from fsspec import filesystem

## Data

In [ ]:
fs = filesystem("https")
ca = gpd.read_parquet(
    "https://github.com/oturns/example_datasets/raw/refs/heads/main/acs/ca_tracts_2021.pq",
    filesystem=fs,
)
cols = [
    "geoid",
    "n_total_households",
    "n_total_pop",
    "n_nonhisp_white_persons",
    "n_nonhisp_black_persons",
    "n_hispanic_persons",
    "n_asian_persons",
    "n_persons_over_60",
    "p_persons_over_60",
    "p_persons_under_18",
    "per_capita_income",
    "median_home_value",
    "median_household_income",
    "p_edu_college_greater",
    "year",
    "geometry",
]

# keep only a subset of columns to keep interactive maps small
ca = ca[cols]
# keep only populated tracts
ca = ca[ca.n_total_pop > 0]
# project into the proper coordinate system
ca = ca.to_crs(32610)

In [ ]:
# san luis obispo
slo = ca[ca.geoid.str.startswith("06079")]
# santa barbara
sb = ca[ca.geoid.str.startswith("06083")]

**The county FIPS code for Monterey county is `06053`. Create a new dataframe called `mont` that stores census tracts for Montery county.**

*Hint: just like above, you want to subset the `ca` dataframe by selecting all rows whose geoid startswith with the Monterey Fips code*

In [ ]:
# your asnwer here

## Visualization

**Create an interactive choropleth map of `median_household_income` using the Santa Barbara dataframe `sb` with the `quantiles` classification scheme and 7 categories.**

*hint: you may want to include `tooltip=False` to avoid the popup obscuring the map view)*

In [ ]:
# your answer here

**Now create an interactive map of the same data (median household income in Santa Barbara county using 7 quantiles) using the colormap `RdBu_r`**

*Hint: the colormap parameter is called `cmap`. See the [week 3](https://knaaptime.com/uppp135-winter26/lectures/week3/geovisualization_inclass.html#colormaps) materials if you forgot how to work with classifications and colormaps*

In [ ]:
# your answer here

**Imagine you are a regional planner writing a report about income disparities in Santa Barbara county. Your goal is to make a map the emphasizes incomes at the top and bottom of the income distribution (i.e. the lowest and highest income places). Which of the maps you created above do you think is the better option for your report? Explain your reasoning.**

## Graphs

In [ ]:
from libpysal.graph import Graph

**In the cells below, create two *queen* graphs (using `Graph.build_contiguity`) for both the `slo` and `sb` dataframes, and name `queen_slo` and `queen_sb`, respectively.**

*hint: see the [week 5](https://knaaptime.com/uppp135-winter26/lectures/week5/graphs_inclass.html) exercise if you forgot how to create contiguity graphs*

**Plot the `slo` dataframe and the `queen_slo` graph in the same interactive map below**

*hint: again you may want to use `tooltip=False`, and see the Week 5 exercise for hints on how to display both layers in the same interactive map*

**Now plot a similar map using the  `sb` dataframe and the `queen_sb` graph in the cells below**

**For both graphs, use the `pct_nonzero` attribute to look at how *dense* each graph is. Which county has a *denser* queen graph?**

*hint: *denser* graphs mean a larger share of percent non-zero entries in the graph*

**Use the `summary()` method to examine summary statistics for the `queen_sb` graph. How many connected components are there in the Graph?**

**For a bonus point, can you explain *why* the `queen_slo` graph has that number of connected components? Hint, the channel islands count as *a single observation***

**Create a *distance band* graph (using `Graph.build_distance_band`) of the `mont` dataframe using a threshold distance of 2000m and save it into a variable called `dist_mont`**

*hint: you will need to set the geometry of the `mont` dataframe to its **centroid**. See class exercises for examples*

**Use the `summary()` method to summarize the `dist_mont` graph. How many *isolates* are present in the graph? What does it mean if an observation is an *isolate*?**

*hint* you may want to try `explore` ing the graph

## Interpolation

In [ ]:
from tobler.area_weighted import area_interpolate
from tobler.util import h3fy

In [ ]:
slo_hex = h3fy(slo, resolution=7)

In [ ]:
slo_hex.explore()

**Use the `area_interpolate` function to send data from our *source*, `slo`, to the *target* hexagons we created above, `slo_hex`. We need two *extensive* variables: "n_total_households" and "n_persons_over_60", as well as two *intensive* variables, "median_home_value" and "per_capita_income"**

**Save the interpolated data into a new variable called `slo_hex_data`**

*hint: see the *week 4* materials in case you forgot how to define extensive and intensive variables*

In [ ]:
# slo_hex_data =

**Using the `slo_hex_data` dataframe, create a *static* map of the variable "n_persons_over_60" using the `fisherjenks` classification scheme, with `6` classes and the `magma` colormap.**

*hint: we use the `plot` method instead of the `explore` method to create a static map instead of an interactive map*

**Using the `slo_hex_data` dataframe, create a *static* map of the variable "per_capita_income" using the `quantiles` classification scheme, with `5` classes and the `RdBu_r` colormap.**

**Using the `slo_hex_data` dataframe, create an *interactive* map of the variable "median_home_value" using the `fisherjenks` classification scheme, with `5` classes and the `viridis` colormap.**

**One problem with simple areal interpolation is it assumes each variable is uniormly-distributed within the source polygons. This can cause estimation problems when your `source` polygons vary in size or cover vast areas of uninhabited land. Is there anything we could do to improve this estimate?**

## Isochrones

In [ ]:
import geosnap as gsp
import pandarm as pdna
from shapely import box

In [ ]:
# create a simple bounding box around Salinas
bbox = gpd.GeoSeries([box(*mont.head(40).total_bounds)], crs=32610).to_frame()
bbox.explore()

In [ ]:
# get an OSM network inside the box
net = pdna.Network.from_gdf(bbox)

In [ ]:
# define a park as our origin point
park = gpd.tools.geocode("Nativiad creek park, salinas ca")
park = park.to_crs(32610)

# create 1500m buffer around the park and view it
buffer_1500m = park.buffer(1500)
buffer_1500m.explore()

**Using the network `net` created above and the `park` dataframe, the geosnap's isochrone function (`gsp.analyze.isochrones_from_gdf`) to create two isochrones, the first extending 1500m from the park, the other extending 2500m.**

**Create an interactive map showing both isochrones. It would be good to make one of them a different color**

**By definition, the distance along a network will always be greater than or equal to the Euclidean (straight-line) distance between any two points. The polygon `buffer_1500m` shows the extent the Euclidean distance 1500m away from the park, meaning the isochrone will always be smaller than `buffer_1500`.**

**How much smaller is it in this example?**

*hint: subtract the area of the 1500m isochrone from the area of `buffer_1500`*

## Segregation

In [ ]:
import segregation as seg

### Single Group

**Use the `seg.singlegroup.Entropy` class to create a Hispanic/non-Hispanic segregation index for each of the three dataframes (`mont`, `sb`, and `slo`) with the group population variable set to "n_hispanic_persons", and the total population variable set to "n_total_pop"**

**Which county has the highest segregation measure according to this index?**

## Multi Group

**Create a multigroup segregation index using the four largest racial/ethnic population groups in California using the `seg.multigroup.MultiInfoTheory` class for Monterey and Santa Barbara counties.**

**Which has a larger segregation index?**

## Geodemographics

In [ ]:
import geosnap as gsp

**Use `gsp.analyze.cluster` to create a geodemographic typology in Santa Barbara county using the variables `"per_capita_income", "median_home_value", "p_persons_under_18", "p_persons_over_60"` with the method `ward` and `5` clusters. Save the result into a new variable called `sb_clusters`**

**Create an interactive map of the `sb_clusters` dataframe that visualizes the `ward` column.**

*hint: to make sure the map is created properly to account for the categorical variable, make sure to set `categorical=True`*

## Conclusion

**What is the single most important or useful thing you learned in this class? If you learned nothing or found it useless, you may say so**

**Add a markdown cell below this one and write your name in** ***bold italic***

**Save your work, then click `File > Download .ipynb` and submit the file on Canvas**